# DEG2MOL Inference Tutorial

This notebook follows the logic in `train.py`, `test.py`, and `inference.py` to generate molecules from DEG profiles selected for a specific target.

Workflow:
1. In the configuration cell below, choose `DATASET_NAME` from `KD`, `KO`, or `Perturb-seq`.
2. Set `TARGET_NAME` to the target you want. In the current data, the target column is `cmap_name`.
3. Run the preview cell to confirm how many rows were selected for that target.
4. Run the inference cell to generate molecules and inspect the output SMILES.
5. You can also provide your own DEG file, align its genes to the model gene order, fill missing genes with zeros, and run inference on it.

Notes:
- The same target can appear in multiple cell lines, so by default the notebook generates molecules for every matched row.
- The notebook includes both a tabular SMILES preview and an RDKit grid visualization for generated molecules.


In [ ]:
from pathlib import Path
import gc
import os
import sys
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw, PandasTools
from torchdiffeq import odeint
from tqdm.auto import tqdm


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (candidate / 'inference.py').exists() and (candidate / 'data').exists():
            return candidate
    raise RuntimeError('Run this notebook from the repository root or from the tutorial directory.')


REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ScafVAE.app.app_utils import ScafVAEBase, load_ModelBase
from models.DEGMON.DEG_AE import GO_Autoencoder
from models.flow.MLP import GatedConditionalFlowMLP

print(f'Repository root: {REPO_ROOT}')
print(f'CUDA available: {torch.cuda.is_available()}')


In [ ]:
DATASET_NAME = 'KO'  # 'KD', 'KO', 'Perturb-seq'
TARGET_NAME = 'AKT1'
CELL_LINE = None      # Example: 'A549'. Use None to include all matched cell lines
MAX_TARGET_ROWS = None  # Example: 3. Use None to keep all matched rows

MODEL_CHECKPOINT = REPO_ROOT / 'checkpoints' / 'DEG2MOL_best_model.pt'
DEG_VAE_CHECKPOINT = REPO_ROOT / 'checkpoints' / 'DEGMON_AE_best_model.pth'
GENE_LIST_PATH = REPO_ROOT / 'data' / 'BP' / 'gene_attribute_matrix_overlap_with_L1000.csv'
TASK_PATH = (REPO_ROOT / '..' / 'ScafVAE' / 'ScafVAE' / 'demo' / 'CMAP_origianl' / 'deg2mol_64dim').resolve()

LATENT_DIM = 64
MODEL_DIM = 512
NUM_LAYERS = 6
COMBINE_METHOD = 'sum'
DROPOUT = 0.0
CONDITIONAL = True
NORMALIZE_CONDITION = False
GUIDANCE_SCALE = 3.0
NUM_SAMPLES = 32
GENERATION_BATCH_SIZE = 32
SOLVER = 'euler'      # 'euler', 'heun', 'rk4', 'dopri5'
NUM_STEPS = 100
SEED = 42

USER_DATA_PATH = None          # Example: REPO_ROOT / 'my_data' / 'custom_deg.csv'
USER_SAMPLE_ID_COLUMN = None  # Example: 'sample_id'. Use None to auto-generate IDs
USER_TARGET_COLUMN = None     # Example: 'target' or 'gene_symbol'
USER_TARGET_VALUE = None      # Example: 'AKT1'. Use None to keep all rows

OUTPUT_DIR = REPO_ROOT / 'tutorial' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
class ODEFunc(torch.nn.Module):
    def __init__(self, flow_model, condition_vector, guidance_scale=1.0, conditional=True):
        super().__init__()
        self.flow_model = flow_model
        self.condition_vector = condition_vector
        self.guidance_scale = guidance_scale
        self.conditional = conditional

    def forward(self, t, x):
        if isinstance(t, float):
            t = torch.tensor([t], device=x.device)
        t_batch = t.expand(x.size(0), 1)

        if self.conditional:
            v_cond = self.flow_model(x, t_batch, self.condition_vector)
            if hasattr(self.flow_model, 'null_condition'):
                null_cond = self.flow_model.null_condition.expand(x.size(0), -1)
                v_uncond = self.flow_model(x, t_batch, null_cond)
                return v_uncond + self.guidance_scale * (v_cond - v_uncond)
            return v_cond

        return self.flow_model(x, t_batch)


def ensure_ready_file(path: Path, description: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f'{description} not found: {path}')
    if path.is_file() and path.stat().st_size == 0:
        raise ValueError(f'{description} is empty: {path}')
    return path


def load_gene_order(gene_list_path: Path):
    gene_order_df = pd.read_csv(gene_list_path, index_col=0)
    return gene_order_df.index.tolist()


def load_target_dataset(dataset_name: str) -> pd.DataFrame:
    dataset_path = REPO_ROOT / 'data' / dataset_name / 'extra_test.feather'
    ensure_ready_file(dataset_path, f'{dataset_name} dataset')
    df = pd.read_feather(dataset_path)
    if 'cmap_name' not in df.columns:
        raise KeyError(f"'cmap_name' column is missing from {dataset_path}")
    return df


def select_target_rows(df: pd.DataFrame, target_name: str, cell_line: str | None = None, max_rows: int | None = None) -> pd.DataFrame:
    target_mask = df['cmap_name'].astype(str).str.upper() == target_name.upper()
    selected = df[target_mask].copy()

    if cell_line is not None:
        if 'cell_iname' not in selected.columns:
            raise KeyError('This dataset does not contain a cell_iname column.')
        selected = selected[selected['cell_iname'].astype(str).str.upper() == cell_line.upper()].copy()

    if selected.empty:
        available_targets = sorted(df['cmap_name'].dropna().astype(str).unique().tolist())
        preview = available_targets[:20]
        raise ValueError(
            f"No rows found for target '{target_name}'"
            + (f" and cell line '{cell_line}'" if cell_line else '')
            + f". Example targets: {preview}"
        )

    selected = selected.reset_index(drop=True)
    if max_rows is not None:
        selected = selected.head(max_rows).copy()
    return selected


def build_deg_tensor(selected_df: pd.DataFrame, gene_order: list[str]) -> torch.Tensor:
    missing_genes = [gene for gene in gene_order if gene not in selected_df.columns]
    if missing_genes:
        raise KeyError(f'Missing genes in dataframe: {missing_genes[:10]}')
    deg_values = selected_df[gene_order].to_numpy(dtype=np.float32)
    return torch.from_numpy(deg_values)


def read_tabular_dataset(path_like) -> pd.DataFrame:
    path = Path(path_like)
    ensure_ready_file(path, 'User dataset')
    suffix = path.suffix.lower()
    if suffix == '.feather':
        return pd.read_feather(path)
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix in {'.csv', '.txt'}:
        return pd.read_csv(path)
    if suffix in {'.tsv', '.tab'}:
        return pd.read_csv(path, sep='\t')
    raise ValueError(f'Unsupported file format: {path}')


def align_user_dataset_to_gene_order(user_df: pd.DataFrame, gene_order: list[str]) -> tuple[pd.DataFrame, list[str], list[str]]:
    aligned_df = user_df.copy()
    present_genes = [gene for gene in gene_order if gene in aligned_df.columns]
    missing_genes = [gene for gene in gene_order if gene not in aligned_df.columns]

    for gene in missing_genes:
        aligned_df[gene] = 0.0

    aligned_df[gene_order] = aligned_df[gene_order].apply(pd.to_numeric, errors='coerce').fillna(0.0).astype(np.float32)
    return aligned_df, present_genes, missing_genes


def prepare_user_dataset(user_data_path, gene_order: list[str], sample_id_column: str | None = None, target_column: str | None = None, target_value: str | None = None) -> tuple[pd.DataFrame, torch.Tensor, dict]:
    raw_df = read_tabular_dataset(user_data_path)
    if target_column is not None and target_value is not None:
        if target_column not in raw_df.columns:
            raise KeyError(f"Column '{target_column}' not found in user dataset")
        mask = raw_df[target_column].astype(str).str.upper() == str(target_value).upper()
        raw_df = raw_df[mask].copy()

    if raw_df.empty:
        raise ValueError('User dataset is empty after filtering.')

    aligned_df, present_genes, missing_genes = align_user_dataset_to_gene_order(raw_df, gene_order)

    if sample_id_column is not None:
        if sample_id_column not in aligned_df.columns:
            raise KeyError(f"Column '{sample_id_column}' not found in user dataset")
        aligned_df['sample_id'] = aligned_df[sample_id_column].astype(str)
    else:
        aligned_df['sample_id'] = [f'user_sample_{i}' for i in range(len(aligned_df))]

    aligned_df = aligned_df.reset_index(drop=True)
    deg_tensor = torch.from_numpy(aligned_df[gene_order].to_numpy(dtype=np.float32))
    stats = {
        'num_rows': len(aligned_df),
        'num_present_genes': len(present_genes),
        'num_missing_genes_filled_with_zero': len(missing_genes),
        'present_gene_examples': present_genes[:10],
        'missing_gene_examples': missing_genes[:10],
    }
    return aligned_df, deg_tensor, stats


def display_molecule_grid(smiles_list, max_mols: int = 5, mols_per_row: int = 5, sub_img_size=(300, 300)):
    valid_smiles = []
    valid_mols = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi) if smi else None
        if mol is not None:
            valid_smiles.append(smi)
            valid_mols.append(mol)
        if len(valid_mols) >= max_mols:
            break

    if not valid_mols:
        print('No valid molecules available for grid visualization.')
        return None

    legends = [f'{idx + 1}: {smi[:40]}' for idx, smi in enumerate(valid_smiles)]
    grid = Draw.MolsToGridImage(valid_mols, molsPerRow=mols_per_row, subImgSize=sub_img_size, legends=legends)
    display(grid)
    return grid


def infer_profile_name(row: pd.Series, row_idx: int) -> str:
    for column in ['sample_id', 'cmap_name', 'Unnamed: 0']:
        if column in row.index and pd.notna(row[column]):
            return str(row[column])
    return f'profile_{row_idx}'


def extract_mu(outputs):
    if isinstance(outputs, tuple):
        if len(outputs) == 4:
            return outputs[1]
        if len(outputs) >= 2:
            return outputs[1]
        return outputs[0]
    return outputs


def load_checkpoint_state(path: Path, device: torch.device):
    checkpoint = torch.load(path, map_location=device)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        return checkpoint['model_state_dict']
    return checkpoint


def load_models(device: torch.device):
    ensure_ready_file(MODEL_CHECKPOINT, 'Flow model checkpoint')
    ensure_ready_file(DEG_VAE_CHECKPOINT, 'DEG encoder checkpoint')
    if not TASK_PATH.exists():
        raise FileNotFoundError(f'TASK_PATH not found: {TASK_PATH}')

    flow_model = GatedConditionalFlowMLP(
        embedding_dim=LATENT_DIM,
        condition_dim=LATENT_DIM,
        model_dim=MODEL_DIM,
        num_layers=NUM_LAYERS,
        combine_method=COMBINE_METHOD,
        dropout=DROPOUT,
    ).to(device)
    flow_model.load_state_dict(torch.load(MODEL_CHECKPOINT, map_location=device)['model_state_dict'])
    flow_model.eval()

    deg_model = GO_Autoencoder(dims=[10280, 2011, 1614, 1075], latent_dim=LATENT_DIM).to(device)
    deg_model.load_state_dict(load_checkpoint_state(DEG_VAE_CHECKPOINT, device))
    deg_model.eval()

    scaf_vae = ScafVAEBase().to(device)
    scaf_checkpoint = load_ModelBase()
    scaf_vae.load_state_dict(scaf_checkpoint['model_state_dict'])
    scaf_vae.eval()

    return flow_model, deg_model, scaf_vae


@torch.no_grad()
def generate_for_targets(selected_df: pd.DataFrame, deg_tensor: torch.Tensor, flow_model, deg_model, scaf_vae, device: torch.device):
    deg_tensor = deg_tensor.to(device)
    mu_deg = extract_mu(deg_model(deg_tensor))

    all_results = []
    iterator = tqdm(range(len(selected_df)), desc='Generating', unit='profile')

    for row_idx in iterator:
        row = selected_df.iloc[row_idx]
        single_mu = mu_deg[row_idx:row_idx + 1]
        generated_smiles = []

        num_chunks = (NUM_SAMPLES + GENERATION_BATCH_SIZE - 1) // GENERATION_BATCH_SIZE
        for chunk_idx in range(num_chunks):
            chunk_start = chunk_idx * GENERATION_BATCH_SIZE
            chunk_end = min(chunk_start + GENERATION_BATCH_SIZE, NUM_SAMPLES)
            current_chunk_size = chunk_end - chunk_start

            if CONDITIONAL:
                z = deg_model.reparameterize(single_mu, None)
                if NORMALIZE_CONDITION:
                    z = F.normalize(z, p=2, dim=1)
                z_condition = z.repeat(current_chunk_size, 1)
                x = torch.randn(current_chunk_size, LATENT_DIM, device=device)
            else:
                z_samples = [deg_model.reparameterize(single_mu, None) for _ in range(current_chunk_size)]
                x = torch.cat(z_samples, dim=0)
                z_condition = None

            if SOLVER == 'dopri5':
                ode_func = ODEFunc(flow_model, z_condition, GUIDANCE_SCALE, CONDITIONAL)
                integration_times = torch.tensor([0.0, 1.0], device=device)
                solution = odeint(ode_func, x, integration_times, method='dopri5', atol=1e-5, rtol=1e-5)
                final_latents = solution[1]
            else:
                dt = 1.0 / NUM_STEPS
                ode_func = ODEFunc(flow_model, z_condition, GUIDANCE_SCALE, CONDITIONAL)
                for step in range(NUM_STEPS):
                    t = torch.tensor([step * dt], device=device)
                    if SOLVER == 'rk4':
                        k1 = ode_func(t, x)
                        k2 = ode_func(t + 0.5 * dt, x + 0.5 * dt * k1)
                        k3 = ode_func(t + 0.5 * dt, x + 0.5 * dt * k2)
                        k4 = ode_func(t + dt, x + dt * k3)
                        x = x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
                    elif SOLVER == 'heun':
                        k1 = ode_func(t, x)
                        k2 = ode_func(t + dt, x + dt * k1)
                        x = x + (dt / 2.0) * (k1 + k2)
                    else:
                        x = x + dt * ode_func(t, x)
                final_latents = x

            decoded = scaf_vae.frag_decoder.sample(
                batch_size=final_latents.size(0),
                input_noise=final_latents,
                output_smi=True,
            )
            generated_smiles.extend(decoded.get('smi', []))

            del x, final_latents, decoded
            if z_condition is not None:
                del z_condition
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        valid_smiles = []
        valid_mols = []
        for smi in generated_smiles:
            if smi in (None, 'None', 'INVALID'):
                continue
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                valid_smiles.append(smi)
                valid_mols.append(mol)

        unique_valid_smiles = sorted(set(valid_smiles))
        all_results.append({
            'target_name': row['cmap_name'] if 'cmap_name' in row.index else None,
            'sample_id': infer_profile_name(row, row_idx),
            'cell_iname': row['cell_iname'] if 'cell_iname' in row.index else None,
            'profile_index': row_idx,
            'num_generated': len(generated_smiles),
            'num_valid': len(valid_smiles),
            'num_unique_valid': len(unique_valid_smiles),
            'generated_smiles': generated_smiles,
            'valid_smiles': valid_smiles,
            'unique_valid_smiles': unique_valid_smiles,
            'valid_mols': valid_mols,
        })

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return all_results


In [ ]:
dataset_df = load_target_dataset(DATASET_NAME)
selected_df = select_target_rows(dataset_df, TARGET_NAME, CELL_LINE, MAX_TARGET_ROWS)
gene_order = load_gene_order(GENE_LIST_PATH)
deg_tensor = build_deg_tensor(selected_df, gene_order)

print(f'Dataset: {DATASET_NAME}')
print(f'Target: {TARGET_NAME}')
print(f'Selected rows: {len(selected_df)}')
print(f'Gene dimension: {deg_tensor.shape[1]}')

meta_columns = [col for col in ['cmap_name', 'cell_iname', 'canonical_smiles', 'Unnamed: 0'] if col in selected_df.columns]
display(selected_df[meta_columns].head(10))


## Optional: User DEG Dataset

The cells below show how to align a user-provided DEG table to the model input format.

Assumptions:
- Each row represents one DEG profile.
- Gene symbols are stored as column names.
- Extra non-gene columns are preserved but are not used for inference.
- Any model-required gene missing from the user file is filled with `0`.


In [ ]:
if USER_DATA_PATH is not None:
    user_aligned_df, user_deg_tensor, user_stats = prepare_user_dataset(
        USER_DATA_PATH,
        gene_order,
        sample_id_column=USER_SAMPLE_ID_COLUMN,
        target_column=USER_TARGET_COLUMN,
        target_value=USER_TARGET_VALUE,
    )
    print(f"User dataset rows: {user_stats['num_rows']}")
    print(f"Matched genes: {user_stats['num_present_genes']}")
    print(f"Missing genes filled with 0: {user_stats['num_missing_genes_filled_with_zero']}")
    print(f"Missing gene examples: {user_stats['missing_gene_examples']}")
    preview_columns = [col for col in ['sample_id', USER_TARGET_COLUMN, 'cmap_name', 'cell_iname'] if col is not None and col in user_aligned_df.columns]
    display(user_aligned_df[preview_columns + gene_order[:5]].head())
else:
    print('Set USER_DATA_PATH first to run custom dataset alignment.')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
flow_model, deg_model, scaf_vae = load_models(device)
results = generate_for_targets(selected_df, deg_tensor, flow_model, deg_model, scaf_vae, device)

summary_df = pd.DataFrame([
    {
        'target_name': item['target_name'],
        'sample_id': item['sample_id'],
        'cell_iname': item['cell_iname'],
        'profile_index': item['profile_index'],
        'num_generated': item['num_generated'],
        'num_valid': item['num_valid'],
        'num_unique_valid': item['num_unique_valid'],
        'example_smiles': item['unique_valid_smiles'][:5],
    }
    for item in results
])

display(summary_df)


In [ ]:
if results:
    first_result = results[0]
    smiles_preview = pd.DataFrame({'smiles': first_result['unique_valid_smiles'][:5]})
    if not smiles_preview.empty:
        PandasTools.AddMoleculeColumnToFrame(smiles_preview, smilesCol='smiles', molCol='molecule')
    display(smiles_preview)
    display_molecule_grid(first_result['unique_valid_smiles'], max_mols=5)

output_path = OUTPUT_DIR / f"{DATASET_NAME}_{TARGET_NAME.replace('/', '_')}_tutorial_results.pkl"
with open(output_path, 'wb') as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved results to: {output_path}')


In [ ]:
if USER_DATA_PATH is not None:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if 'flow_model' not in globals() or 'deg_model' not in globals() or 'scaf_vae' not in globals():
        flow_model, deg_model, scaf_vae = load_models(device)

    user_results = generate_for_targets(user_aligned_df, user_deg_tensor, flow_model, deg_model, scaf_vae, device)
    user_summary_df = pd.DataFrame([
        {
            'sample_id': item['sample_id'],
            'target_name': item['target_name'],
            'cell_iname': item['cell_iname'],
            'profile_index': item['profile_index'],
            'num_generated': item['num_generated'],
            'num_valid': item['num_valid'],
            'num_unique_valid': item['num_unique_valid'],
            'example_smiles': item['unique_valid_smiles'][:5],
        }
        for item in user_results
    ])
    display(user_summary_df)

    user_output_label = Path(USER_DATA_PATH).stem
    if USER_TARGET_VALUE is not None:
        safe_target_value = str(USER_TARGET_VALUE).replace('/', '_')
        user_output_label = f'{user_output_label}_{safe_target_value}'
    user_output_path = OUTPUT_DIR / f'{user_output_label}_tutorial_results.pkl'
    with open(user_output_path, 'wb') as f:
        pickle.dump(user_results, f, protocol=pickle.HIGHEST_PROTOCOL)

    if user_results:
        user_preview_df = pd.DataFrame({'smiles': user_results[0]['unique_valid_smiles'][:5]})
        if not user_preview_df.empty:
            PandasTools.AddMoleculeColumnToFrame(user_preview_df, smilesCol='smiles', molCol='molecule')
        display(user_preview_df)
        display_molecule_grid(user_results[0]['unique_valid_smiles'], max_mols=5)

    print(f'Saved user dataset results to: {user_output_path}')
else:
    print('Set USER_DATA_PATH first to run custom dataset inference.')
